In [4]:
#Import libraries and load dataset
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

from src.config import (
    CLEANED_DATA_PATH,
    RESULTS_DIR,
    COUNTRY_ZTEST_PATH,
    COUNTRY_BOOTSTRAP_PATH,
    COUNTRY_FINAL_PATH
)

print("Project root:", PROJECT_ROOT)
print("Cleaned data path:", CLEANED_DATA_PATH)
print("Cleaned data exists:", CLEANED_DATA_PATH.exists())

Project root: d:\2.DAP391m_Project
Cleaned data path: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
Cleaned data exists: True


In [5]:
df = pd.read_csv(CLEANED_DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

# Check infomation
display(df.head())
print(df["country"].value_counts())
print(df["landing_page"].value_counts())
print(df["converted"].value_counts())

Dataset loaded successfully!
Shape: (288541, 9)


,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


country
US    202186
UK     71961
CA     14394
Name: count, dtype: int64
landing_page
new_page    144315
old_page    144226
Name: count, dtype: int64
converted
0    254058
1     34483
Name: count, dtype: int64


## Define Country-Level Z-test Function

This function performs a two-proportion z-test within a single country.

For each country, the null and alternative hypotheses are:

H0: CR_new_country = CR_old_country  
H1: CR_new_country != CR_old_country

The test checks whether the conversion rate difference between old_page and new_page is statistically significant within that country.

In [7]:
def analyze_country_ztest(df, country_name):
    # Filter data for one country
    country_data = df[df["country"] == country_name]

    # Split country data into old_page and new_page
    old_page_data = country_data[country_data["landing_page"] == "old_page"]
    new_page_data = country_data[country_data["landing_page"] == "new_page"]

    # Count users
    old_users = old_page_data["user_id"].count()
    new_users = new_page_data["user_id"].count()

    # Count conversions
    old_conversions = old_page_data["converted"].sum()
    new_conversions = new_page_data["converted"].sum()

    # Calculate conversion rates
    old_cr = old_conversions / old_users
    new_cr = new_conversions / new_users

    # Calculate lift and uplift
    absolute_lift_pp = (new_cr - old_cr) * 100

    relative_uplift_percent = (
        (new_cr - old_cr) / old_cr
    ) * 100

    # Prepare inputs for two-proportion z-test
    count = [
        new_conversions,
        old_conversions
    ]

    nobs = [
        new_users,
        old_users
    ]

    # Run two-sided z-test
    z_stat, p_value = proportions_ztest(
        count=count,
        nobs=nobs,
        alternative="two-sided"
    )

    # Store result
    result = {
        "country": country_name,
        "old_users": old_users,
        "new_users": new_users,
        "old_conversions": old_conversions,
        "new_conversions": new_conversions,
        "old_cr_percent": old_cr * 100,
        "new_cr_percent": new_cr * 100,
        "absolute_lift_pp": absolute_lift_pp,
        "relative_uplift_percent": relative_uplift_percent,
        "z_statistic": z_stat,
        "p_value": p_value,
        "decision": "Reject H0" if p_value < 0.05 else "Fail to reject H0"
    }

    return result

## Run Z-test for Each Country

We apply the country-level z-test function separately to CA, UK, and US.

This allows us to check whether the observed country-level differences are statistically significant.

In [8]:
ca_result = analyze_country_ztest(df, "CA")
uk_result = analyze_country_ztest(df, "UK")
us_result = analyze_country_ztest(df, "US")

country_ztest_results = pd.DataFrame([
    ca_result,
    uk_result,
    us_result
])

country_ztest_results

,country,old_users,new_users,old_conversions,new_conversions,old_cr_percent,new_cr_percent,absolute_lift_pp,relative_uplift_percent,z_statistic,p_value,decision
0,CA,7138,7256,848,811,11.880078,11.176957,-0.703121,-5.918492,-1.320795,0.186570,Fail to reject H0
1,UK,36100,35861,4330,4341,11.994460,12.105072,0.110613,0.922197,0.455738,0.648578,Fail to reject H0
2,US,100988,101198,12171,11982,12.051927,11.840155,-0.211772,-1.757163,-1.468011,0.142101,Fail to reject H0


## Apply Bonferroni Correction

Since we conduct multiple hypothesis tests across countries, the chance of false positives increases.

To adjust for multiple comparisons, we apply Bonferroni correction:

alpha_corrected = alpha / number_of_tests

Here, we test 3 countries, so:

alpha_corrected = 0.05 / 3 = 0.0167

In [9]:
alpha = 0.05
number_of_tests = country_ztest_results.shape[0]
alpha_corrected = alpha / number_of_tests

country_ztest_results["alpha"] = alpha
country_ztest_results["alpha_corrected"] = alpha_corrected

country_ztest_results["decision_bonferroni"] = country_ztest_results["p_value"].apply(
    lambda p: "Reject H0" if p < alpha_corrected else "Fail to reject H0"
)

country_ztest_results

,country,old_users,new_users,old_conversions,new_conversions,old_cr_percent,new_cr_percent,absolute_lift_pp,relative_uplift_percent,z_statistic,p_value,decision,alpha,alpha_corrected,decision_bonferroni
0,CA,7138,7256,848,811,11.880078,11.176957,-0.703121,-5.918492,-1.320795,0.186570,Fail to reject H0,0.05,0.016667,Fail to reject H0
1,UK,36100,35861,4330,4341,11.994460,12.105072,0.110613,0.922197,0.455738,0.648578,Fail to reject H0,0.05,0.016667,Fail to reject H0
2,US,100988,101198,12171,11982,12.051927,11.840155,-0.211772,-1.757163,-1.468011,0.142101,Fail to reject H0,0.05,0.016667,Fail to reject H0


## Define Country-Level Bootstrap CI Function

The country-level z-test gives a p-value for each country.

To estimate the uncertainty of the uplift within each country, we also compute bootstrap confidence intervals.

For each country:

1. Filter users in that country.
2. Split users into old_page and new_page.
3. Resample old_page and new_page separately with replacement.
4. Calculate uplift for each bootstrap sample.
5. Repeat the process 10,000 times.
6. Calculate the 95% confidence interval using the 2.5th and 97.5th percentiles.

In [10]:
def bootstrap_country_uplift_ci(
    df,
    country_name,
    n_bootstrap=10000,
    random_state=42
):
    # Set random seed for reproducibility
    np.random.seed(random_state)

    # Filter data for one country
    country_data = df[df["country"] == country_name]

    # Split into old_page and new_page groups
    old_page_data = country_data[country_data["landing_page"] == "old_page"]
    new_page_data = country_data[country_data["landing_page"] == "new_page"]

    # Get sample sizes
    old_n = len(old_page_data)
    new_n = len(new_page_data)

    # Calculate original conversion rates
    old_cr = old_page_data["converted"].mean()
    new_cr = new_page_data["converted"].mean()

    # Calculate original uplift
    original_uplift = (
        (new_cr - old_cr) / old_cr
    ) * 100

    # Store bootstrap uplift values
    bootstrap_uplifts = []

    for i in range(n_bootstrap):
        # Sample with replacement within each country-page group
        old_sample = old_page_data.sample(
            n=old_n,
            replace=True
        )

        new_sample = new_page_data.sample(
            n=new_n,
            replace=True
        )

        # Calculate bootstrap conversion rates
        old_sample_cr = old_sample["converted"].mean()
        new_sample_cr = new_sample["converted"].mean()

        # Calculate bootstrap uplift
        sample_uplift = (
            (new_sample_cr - old_sample_cr)
            / old_sample_cr
        ) * 100

        bootstrap_uplifts.append(sample_uplift)

    bootstrap_uplifts = np.array(bootstrap_uplifts)

    # Calculate bootstrap summary
    mean_bootstrap_uplift = bootstrap_uplifts.mean()
    ci_lower = np.percentile(bootstrap_uplifts, 2.5)
    ci_upper = np.percentile(bootstrap_uplifts, 97.5)

    result = {
        "country": country_name,
        "old_users": old_n,
        "new_users": new_n,
        "old_cr_percent": old_cr * 100,
        "new_cr_percent": new_cr * 100,
        "original_uplift": original_uplift,
        "mean_bootstrap_uplift": mean_bootstrap_uplift,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "n_bootstrap": n_bootstrap,
        "ci_contains_zero": ci_lower <= 0 <= ci_upper
    }

    return result

## Run Bootstrap CI for Each Country

We run country-level bootstrap confidence intervals for CA, UK, and US.

The output shows the original uplift, mean bootstrap uplift, and 95% confidence interval for each country.

In [11]:
ca_bootstrap = bootstrap_country_uplift_ci(
    df,
    country_name="CA",
    n_bootstrap=10000,
    random_state=42
)

uk_bootstrap = bootstrap_country_uplift_ci(
    df,
    country_name="UK",
    n_bootstrap=10000,
    random_state=42
)

us_bootstrap = bootstrap_country_uplift_ci(
    df,
    country_name="US",
    n_bootstrap=10000,
    random_state=42
)

country_bootstrap_summary = pd.DataFrame([
    ca_bootstrap,
    uk_bootstrap,
    us_bootstrap
])

country_bootstrap_summary

,country,old_users,new_users,old_cr_percent,new_cr_percent,original_uplift,mean_bootstrap_uplift,ci_lower,ci_upper,n_bootstrap,ci_contains_zero
0,CA,7138,7256,11.880078,11.176957,-5.918492,-5.820678,-14.291513,3.110459,10000,True
1,UK,36100,35861,11.994460,12.105072,0.922197,0.926684,-2.997456,4.951255,10000,True
2,US,100988,101198,12.051927,11.840155,-1.757163,-1.758412,-4.092975,0.619185,10000,True


## Combine Z-test and Bootstrap Results

To make interpretation easier, we combine the country-level z-test results with the country-level bootstrap confidence intervals.

The final table includes:

- conversion rate by country
- relative uplift
- z-test p-value
- Bonferroni-adjusted decision
- bootstrap confidence interval
- final interpretation

In [12]:
country_final_results = country_ztest_results.merge(
    country_bootstrap_summary[
        [
            "country",
            "original_uplift",
            "mean_bootstrap_uplift",
            "ci_lower",
            "ci_upper",
            "n_bootstrap",
            "ci_contains_zero"
        ]
    ],
    on="country",
    how="left"
)

country_final_results

,country,old_users,new_users,old_conversions,new_conversions,old_cr_percent,new_cr_percent,absolute_lift_pp,relative_uplift_percent,z_statistic,...,decision,alpha,alpha_corrected,decision_bonferroni,original_uplift,mean_bootstrap_uplift,ci_lower,ci_upper,n_bootstrap,ci_contains_zero
0,CA,7138,7256,848,811,11.880078,11.176957,-0.703121,-5.918492,-1.320795,...,Fail to reject H0,0.05,0.016667,Fail to reject H0,-5.918492,-5.820678,-14.291513,3.110459,10000,True
1,UK,36100,35861,4330,4341,11.994460,12.105072,0.110613,0.922197,0.455738,...,Fail to reject H0,0.05,0.016667,Fail to reject H0,0.922197,0.926684,-2.997456,4.951255,10000,True
2,US,100988,101198,12171,11982,12.051927,11.840155,-0.211772,-1.757163,-1.468011,...,Fail to reject H0,0.05,0.016667,Fail to reject H0,-1.757163,-1.758412,-4.092975,0.619185,10000,True


In [13]:
# Add Final Interpretation
def interpret_country_result(row):
    if row["p_value"] < row["alpha_corrected"] and row["ci_contains_zero"] == False:
        if row["relative_uplift_percent"] > 0:
            return "New page performs significantly better."
        else:
            return "New page performs significantly worse."
    else:
        return "No statistically significant difference."

country_final_results["final_interpretation"] = country_final_results.apply(
    interpret_country_result,
    axis=1
)

country_final_results

,country,old_users,new_users,old_conversions,new_conversions,old_cr_percent,new_cr_percent,absolute_lift_pp,relative_uplift_percent,z_statistic,...,alpha,alpha_corrected,decision_bonferroni,original_uplift,mean_bootstrap_uplift,ci_lower,ci_upper,n_bootstrap,ci_contains_zero,final_interpretation
0,CA,7138,7256,848,811,11.880078,11.176957,-0.703121,-5.918492,-1.320795,...,0.05,0.016667,Fail to reject H0,-5.918492,-5.820678,-14.291513,3.110459,10000,True,No statistically significant difference.
1,UK,36100,35861,4330,4341,11.994460,12.105072,0.110613,0.922197,0.455738,...,0.05,0.016667,Fail to reject H0,0.922197,0.926684,-2.997456,4.951255,10000,True,No statistically significant difference.
2,US,100988,101198,12171,11982,12.051927,11.840155,-0.211772,-1.757163,-1.468011,...,0.05,0.016667,Fail to reject H0,-1.757163,-1.758412,-4.092975,0.619185,10000,True,No statistically significant difference.


## Save Country-Level Results

Three result files are saved:

1. `country_ztest_results.csv`  
   Contains z-test results and Bonferroni-corrected decisions.

2. `country_bootstrap_summary.csv`  
   Contains country-level bootstrap confidence intervals.

3. `country_final_results.csv`  
   Combines z-test results, bootstrap confidence intervals, and final interpretation.

In [14]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

country_ztest_results.to_csv(
    COUNTRY_ZTEST_PATH,
    index=False
)

country_bootstrap_summary.to_csv(
    COUNTRY_BOOTSTRAP_PATH,
    index=False
)

country_final_results.to_csv(
    COUNTRY_FINAL_PATH,
    index=False
)

print(COUNTRY_ZTEST_PATH.exists())
print(COUNTRY_BOOTSTRAP_PATH.exists())
print(COUNTRY_FINAL_PATH.exists())

True
True
True


## Interpretation

Country-level hypothesis testing was conducted to check whether the conversion rate difference between `old_page` and `new_page` is statistically significant within each country.

The results show that none of the country-level p-values are below the 0.05 significance threshold. After Bonferroni correction, all country-level tests still fail to reject the null hypothesis.

Country-level bootstrap confidence intervals were also computed to estimate the uncertainty of uplift in each market. These intervals help evaluate whether the observed uplift is robust or whether it may be explained by sampling variation.

Overall, the country-level analysis shows that:

- UK has a small positive descriptive uplift.
- US has a negative descriptive uplift.
- CA has the largest negative descriptive uplift and the largest downside risk.

However, none of these country-level differences are statistically significant. Therefore, the country-level results should be interpreted cautiously and mainly as exploratory market-level insights.